# 04 — Predict Upcoming NBA Games

In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
from pathlib import Path

# Load artifacts
rf = joblib.load('../models/random_forest.pkl')
feature_cols = joblib.load('../models/model_features.pkl')

team_games = pd.read_csv('../data/modelling/team_games.csv')
team_games['game_date'] = pd.to_datetime(team_games['game_date'], errors='coerce')

# Schedule
SCHEDULE_URL = 'https://cdn.nba.com/static/json/staticData/scheduleLeagueV2.json'
data = requests.get(SCHEDULE_URL).json()
rows = []
for gdate in data['leagueSchedule']['gameDates']:
    date = gdate['gameDate']
    for g in gdate['games']:
        home = g['homeTeam']['teamName']
        visitor = g['awayTeam']['teamName']
        rows.append({'Date': date, 'Home': home, 'Visitor': visitor})

schedule = pd.DataFrame(rows)
schedule['Date'] = pd.to_datetime(schedule['Date'], errors='coerce')

# Only upcoming
today = pd.Timestamp.today().normalize()
upcoming = schedule[schedule['Date'] >= today].sort_values('Date').reset_index(drop=True)

# Team name normalization
NICK_TO_FULL = {
    'Hawks':'Atlanta Hawks','Celtics':'Boston Celtics','Nets':'Brooklyn Nets','Hornets':'Charlotte Hornets',
    'Bulls':'Chicago Bulls','Cavaliers':'Cleveland Cavaliers','Mavericks':'Dallas Mavericks','Nuggets':'Denver Nuggets',
    'Pistons':'Detroit Pistons','Warriors':'Golden State Warriors','Rockets':'Houston Rockets','Pacers':'Indiana Pacers',
    'Clippers':'LA Clippers','LA Clippers':'LA Clippers','Los Angeles Clippers':'LA Clippers','Lakers':'Los Angeles Lakers',
    'Grizzlies':'Memphis Grizzlies','Heat':'Miami Heat','Bucks':'Milwaukee Bucks','Timberwolves':'Minnesota Timberwolves',
    'Pelicans':'New Orleans Pelicans','Knicks':'New York Knicks','Thunder':'Oklahoma City Thunder','Magic':'Orlando Magic',
    '76ers':'Philadelphia 76ers','Suns':'Phoenix Suns','Trail Blazers':'Portland Trail Blazers','Kings':'Sacramento Kings',
    'Spurs':'San Antonio Spurs','Raptors':'Toronto Raptors','Jazz':'Utah Jazz'
}

DATASET_TEAMS = set(team_games['team_name'].unique())

# Mapping helper

def to_dataset_name(name: str):
    if name in DATASET_TEAMS:
        return name
    if name in NICK_TO_FULL:
        full = NICK_TO_FULL[name]
        if full == 'LA Clippers' and 'LA Clippers' not in DATASET_TEAMS and 'Los Angeles Clippers' in DATASET_TEAMS:
            return 'Los Angeles Clippers'
        if full in DATASET_TEAMS:
            return full
    for ds in DATASET_TEAMS:
        if ds.endswith(name):
            return ds
    for ds in DATASET_TEAMS:
        if name in ds:
            return ds
    return None

# Latest features

def latest_team_features(team_name: str) -> pd.DataFrame:
    df = team_games[team_games['team_name'] == team_name].copy()
    if df.empty:
        return pd.DataFrame()
    df = df.sort_values('game_date')
    latest = df.tail(1).copy()
    latest = latest.drop(columns=['home_away_flag'], errors='ignore')
    return latest

# Build row

def build_game_features(visitor_sched: str, home_sched: str):
    home_name = to_dataset_name(home_sched)
    away_name = to_dataset_name(visitor_sched)
    if home_name is None or away_name is None:
        return None
    home_latest = latest_team_features(home_name).add_prefix('home_')
    away_latest = latest_team_features(away_name).add_prefix('away_')
    if home_latest.empty or away_latest.empty:
        return None
    row = pd.concat([home_latest, away_latest], axis=1)
    for c in feature_cols:
        if c.startswith('diff_'):
            base = c.replace('diff_','')
            hc, ac = 'home_stat_' + base, 'away_stat_' + base
            row[c] = 0.0
            if hc in row.columns and ac in row.columns:
                row[c] = pd.to_numeric(row[hc], errors='coerce') - pd.to_numeric(row[ac], errors='coerce')
    row = row.reindex(columns=feature_cols, fill_value=0)
    return row

pred_rows = []
for _, g in upcoming.iterrows():
    date = g['Date']
    home = g['Home']
    visitor = g['Visitor']
    feats = build_game_features(visitor, home)
    if feats is None:
        continue
    proba = rf.predict_proba(feats)[0][1]
    pred = rf.predict(feats)[0]
    pred_rows.append({
        'Date': date.date(),
        'Visitor': visitor,
        'Home': home,
        'Home_Win_Probability (%)': round(proba*100, 1),
        'Predicted_Winner': home if pred == 1 else visitor
    })

pred_df = pd.DataFrame(pred_rows)
print(pred_df.head())

out_path = Path('../outputs/predictions/upcoming_predictions.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(out_path, index=False)
print(f"Saved predictions → {out_path} ({len(pred_df)} games)")
